# Evaluación de Modelos

---

## Descripción

Este notebook evalúa todos los modelos entrenados en [02_supervised_modeling.ipynb](02_supervised_modeling.ipynb).
Se comparan modelos de regresión (LinearRegression, DecisionTreeRegressor)
y modelos de clasificación (DecisionTreeClassifier, LogisticRegression, SVM).

---

## Requisitos de Software

- pandas (>=1.1.0)
- numpy (>=2.0.0)
- scikit-learn (>=1.3)
- matplotlib (>=3.7.1)
- seaborn (>=0.12.0)

In [2]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, mean_absolute_error,
    r2_score, mean_squared_error
)

In [3]:
data = pd.read_csv('../data/dataset_clientes.csv')
data.head()

,id_cliente,fecha_registro,edad,genero,region,estado_civil,ingreso_mensual,gasto_mensual,deuda_total,score_crediticio,...,ultima_compra_dias,uso_app,tipo_plan,num_productos,tiene_tarjeta_credito,canal_registro,dia_semana_registro,hora_registro,codigo_postal,abandono
0,1,2021-10-27,66,Otro,Norte,Divorciado,9.243057e+05,524088.303055,2.448145e+06,455.406680,...,356,Bajo,Estandar,3,1,Tienda,Lunes,22,3824,1
1,2,2018-08-25,51,Masculino,Centro,Soltero,1.384687e+06,314259.751474,1.620569e+06,575.048508,...,307,Medio,Premium,4,1,App,Martes,10,4148,0
2,3,2019-05-25,48,Femenino,Norte,Casado,NaN,387192.316142,5.395040e+06,770.716904,...,232,Alto,Premium,4,1,App,Jueves,6,7200,0
3,4,2022-04-20,54,Masculino,Sur,Casado,4.369032e+05,417328.601856,2.999350e+06,442.722671,...,165,Alto,Estandar,2,1,App,Domingo,16,1782,1
4,5,2020-03-19,31,Otro,Centro,Soltero,7.408561e+05,490961.191253,1.637711e+06,468.188403,...,283,Bajo,Estandar,3,1,Web,Martes,8,3448,1


# Preparación

In [4]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Tratamiento de atípicos via recorte por percentiles.
    """
    def __init__(self, limits=(0.05, 0.05)):
        self.limits = limits

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.columns_ = X.columns
        else:
            self.columns_ = np.arange(X.shape[1])
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_)
        for col in self.columns_:
            lower = X[col].quantile(self.limits[0])
            upper = X[col].quantile(1 - self.limits[1])
            X = X.astype('float64')
            X[col] = np.clip(X[col], lower, upper)
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array(self.columns_)
        return np.array(input_features)


def tratar_duplicados(X: pd.DataFrame, drop: bool = True) -> pd.DataFrame:
    """
    Tratamiento de duplicados.
    Si drop=True elimina filas duplicadas, si no las deja.
    """
    return X.drop_duplicates() if drop else X


class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Elimina variables con alta correlación (multicolinealidad).
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.columns_to_drop_ = None

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        corr_matrix = X_df.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.columns_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df.drop(columns=self.columns_to_drop_, errors='ignore').values


class DataFrameConverter(BaseEstimator, TransformerMixin):
    """
    Convierte el array de ColumnTransformer en DataFrame con nombres de columnas.
    """
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        self.feature_names_ = None

    def fit(self, X, y=None):
        self.feature_names_ = self.preprocessor.get_feature_names_out()
        return self

    def transform(self, X):
        return pd.DataFrame(X, columns=self.feature_names_)

In [5]:
def evaluar(modelo, X_train, X_test, y_train, y_test):
    """
    Entrena el modelo y retorna métricas de clasificación.
    """
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    return {
        'accuracy':  accuracy_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob)
    }

## Evaluación de modelos de regresión

> Variable objetivo: score_crediticio

In [6]:
target_reg = 'score_crediticio'

features_num = [
    'edad', 'ingreso_mensual', 'gasto_mensual', 'deuda_total',
    'antiguedad_meses', 'frecuencia_compra', 'ultima_compra_dias', 'num_productos'
]
features_cat = [
    'genero', 'region', 'estado_civil', 'uso_app', 'tipo_plan', 'canal_registro'
]

X_reg = data[features_num + features_cat]
y_reg = data[target_reg]

mask = y_reg.notna()
X_reg, y_reg = X_reg[mask], y_reg[mask]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=28
)

In [7]:
numeric_transformer = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

In [8]:
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

modelo_lr = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor_lr),
    ('conversion',    DataFrameConverter(preprocessor_lr)),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        LinearRegression())
])

modelo_dtr = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('conversion',    DataFrameConverter(preprocessor)),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        DecisionTreeRegressor(
                          max_depth=3, min_samples_leaf=10,
                          min_samples_split=100, ccp_alpha=0.01, random_state=29))
])

modelo_lr.fit(X_train_reg, y_train_reg)
modelo_dtr.fit(X_train_reg, y_train_reg)

y_pred_lr  = modelo_lr.predict(X_test_reg)
y_pred_dtr = modelo_dtr.predict(X_test_reg)

resultados_reg = pd.DataFrame({
    'Modelo': ['LinearRegression', 'DecisionTreeRegressor'],
    'R2':   [r2_score(y_test_reg, y_pred_lr),   r2_score(y_test_reg, y_pred_dtr)],
    'MAE':  [mean_absolute_error(y_test_reg, y_pred_lr),  mean_absolute_error(y_test_reg, y_pred_dtr)],
    'RMSE': [np.sqrt(mean_squared_error(y_test_reg, y_pred_lr)),
             np.sqrt(mean_squared_error(y_test_reg, y_pred_dtr))]
})

resultados_reg.set_index('Modelo', inplace=True)
print(resultados_reg.to_string())

                             R2        MAE       RMSE
Modelo                                               
LinearRegression       0.000595  79.212815  99.454627
DecisionTreeRegressor  0.001090  79.119265  99.430005


## Evaluación de modelos de clasificación

> Variable objetivo: abandono

In [9]:
target_cls = 'abandono'

X_cls = data[features_num + features_cat]
y_cls = data[target_cls]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=29, stratify=y_cls
)

In [12]:
pipeline_dtc = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        DecisionTreeClassifier(random_state=29))
])

pipeline_logreg = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        LogisticRegression(max_iter=10000, random_state=29))
])

metricas_dtc    = evaluar(pipeline_dtc,    X_train_cls, X_test_cls, y_train_cls, y_test_cls)
metricas_logreg = evaluar(pipeline_logreg, X_train_cls, X_test_cls, y_train_cls, y_test_cls)

resultados_cls = pd.DataFrame({
    'Modelo': ['DecisionTreeClassifier', 'LogisticRegression'],
    'Accuracy':  [metricas_dtc['accuracy'],  metricas_logreg['accuracy']],
    'F1':        [metricas_dtc['f1'],         metricas_logreg['f1']],
    'Precision': [metricas_dtc['precision'],  metricas_logreg['precision']],
    'Recall':    [metricas_dtc['recall'],     metricas_logreg['recall']],
    'ROC AUC':   [metricas_dtc['roc_auc'],    metricas_logreg['roc_auc']]
})

resultados_cls.set_index('Modelo', inplace=True)
print(resultados_cls.to_string())

                        Accuracy        F1  Precision    Recall   ROC AUC
Modelo                                                                   
DecisionTreeClassifier  0.568873  0.450827   0.455808  0.445954  0.547845
LogisticRegression      0.648775  0.452008   0.593373  0.365040  0.666643
